In [ ]:
!pip install bacformer

In [ ]:
# Load model directly
from transformers import AutoModelForMaskedLM
model = AutoModelForMaskedLM.from_pretrained("macwiatrak/bacformer-masked-complete-genomes", trust_remote_code=True, dtype="auto")

In [ ]:
import os
os.listdir(".")

In [ ]:
!pip -q install transformers biopython umap-learn plotly pandas scikit-learn

In [ ]:
import re
import gzip
import torch
import numpy as np
import pandas as pd
from Bio import SeqIO
from transformers import AutoModel
from collections import Counter, defaultdict
import plotly.express as px
import umap.umap_ as umap

In [ ]:
GFF_PATH = "GCF_000069245.1.gff"
FAA_PATH = "ABAYE.1_proteins.faa"
ANNOT_PATH = "ABAYE.1_panggo.emapper.annotations"

In [ ]:
def open_maybe_gz(path: str, mode: str = "rt"):
    return gzip.open(path, mode) if path.endswith(".gz") else open(path, mode, encoding="utf-8", errors="replace")

def parse_gff_attrs(attr_str: str) -> dict:
    out = {}
    for item in attr_str.strip().split(";"):
        item = item.strip()
        if not item:
            continue
        if "=" in item:
            k, v = item.split("=", 1)
            out[k.strip()] = v.strip()
    return out

def norm_faa_id(s: str) -> str:
    tok = s.split()[0]
    return tok.split("|")[-1]

def parse_ppanggolin_gff_cds(gff_path: str) -> pd.DataFrame:
    rows = []
    contig_counts = Counter()

    with open_maybe_gz(gff_path, "rt") as f:
        for line in f:
            if not line or line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9 or parts[2] != "CDS":
                continue

            seqid, source, ftype, start, end, score, strand, phase, attrs_str = parts
            a = parse_gff_attrs(attrs_str)

            rows.append(
                {
                    "seqid": seqid,
                    "start": int(start),
                    "end": int(end),
                    "strand": strand,
                    "ID": a.get("ID", ""),
                    "Parent": a.get("Parent", ""),
                    "family": a.get("family", ""),
                    "partition": a.get("partition", ""),
                    "rgp": a.get("rgp", ""),
                    "module": a.get("module", ""),
                    "spot": a.get("spot", ""),
                    "product": a.get("product", ""),
                }
            )
            contig_counts[seqid] += 1

    df = pd.DataFrame(rows).sort_values(["seqid", "start", "end", "ID"]).reset_index(drop=True)
    print("GFF CDS:", len(df), "| contigs:", df["seqid"].nunique(), "| top contigs:", contig_counts.most_common(5))
    print("GFF missing family:", int((df["family"] == "").sum()))
    return df

def load_faa_queues_by_family(faa_path: str) -> dict[str, list[tuple[str, str]]]:
    fam2q = defaultdict(list)
    total = 0

    with open_maybe_gz(faa_path, "rt") as f:
        for rec in SeqIO.parse(f, "fasta"):
            total += 1
            fam = norm_faa_id(str(rec.id))  # FASTA id = family
            fam2q[fam].append((str(rec.id), str(rec.seq)))

    if total == 0:
        raise ValueError("FAA vide ou non lisible.")

    print("FAA records:", total, "| unique families in FAA:", len(fam2q))
    print("Example FAA ids:", list(fam2q.keys())[:5])
    return fam2q

def match_gff_to_faa_by_family(gff_df: pd.DataFrame, fam2q: dict[str, list[tuple[str, str]]]) -> pd.DataFrame:
    df = gff_df.copy()
    df["faa_id"] = ""
    df["protein_sequence"] = ""
    stats = Counter()

    queues = {k: list(v) for k, v in fam2q.items()}

    for i, r in df.iterrows():
        fam = r["family"]
        if not fam:
            stats["gff_no_family"] += 1
            continue
        if fam not in queues:
            stats["family_not_in_faa"] += 1
            continue
        if not queues[fam]:
            stats["family_exhausted"] += 1
            continue

        faa_id, seq = queues[fam].pop(0)
        df.at[i, "faa_id"] = faa_id
        df.at[i, "protein_sequence"] = seq
        stats["matched"] += 1

    print("Match stats:", dict(stats))
    print("Matched:", int((df["protein_sequence"] != "").sum()), "/", len(df))
    return df

def build_bacformer_genome_info(matched_df: pd.DataFrame):
    df = matched_df[matched_df["protein_sequence"] != ""].copy()
    df = df.sort_values(["seqid", "start", "end", "ID"]).reset_index(drop=True)

    contigs = list(df["seqid"].unique())
    contig_to_idx = {c: i for i, c in enumerate(contigs)}
    df["contig_idx"] = df["seqid"].map(contig_to_idx).astype(int)

    df = df.sort_values(["contig_idx", "start", "end", "ID"]).reset_index(drop=True)
    df["protein_idx_flat"] = np.arange(len(df))

    prot_seqs_by_contig = [[] for _ in contigs]
    prot_ids_by_contig  = [[] for _ in contigs]

    for _, r in df.iterrows():
        ci = int(r["contig_idx"])
        prot_seqs_by_contig[ci].append(r["protein_sequence"])
        prot_ids_by_contig[ci].append(r["ID"] if r["ID"] else r["faa_id"])

    genome_info = {
        "contig_ids": contigs,
        "protein_sequence": prot_seqs_by_contig,
        "protein_id": prot_ids_by_contig,
    }
    return genome_info, df

# RUN
gff_df = parse_ppanggolin_gff_cds(GFF_PATH)
fam2q = load_faa_queues_by_family(FAA_PATH)

print("Families intersection:", len(set(gff_df["family"]) & set(fam2q.keys())))

matched_df = match_gff_to_faa_by_family(gff_df, fam2q)
genome_info, genes_df = build_bacformer_genome_info(matched_df)

print("Final proteins:", len(genes_df), "| contigs:", len(genome_info["contig_ids"]))
genes_df.head()

In [ ]:
# compte familles dans GFF
gff_family_counts = Counter(gff_df["family"].tolist())

# compte familles dans FAA (nb de séquences par family = longueur de la queue)
faa_family_counts = {fam: len(q) for fam, q in fam2q.items()}

# familles exhausted = gff_count > faa_count
exhausted = []
for fam, gff_c in gff_family_counts.items():
    faa_c = faa_family_counts.get(fam, 0)
    if gff_c > faa_c:
        exhausted.append((fam, gff_c, faa_c, gff_c - faa_c))

exhausted_df = pd.DataFrame(exhausted, columns=["family", "gff_count", "faa_count", "missing"]).sort_values(
    ["missing", "gff_count"], ascending=False
)

print("n exhausted families:", len(exhausted_df), " | total missing CDS:", int(exhausted_df["missing"].sum()))
exhausted_df.head(30)


In [ ]:
missing_rows = matched_df[matched_df["protein_sequence"] == ""].copy()
missing_rows[["seqid","start","end","ID","family","partition","rgp","module","product"]].head(50)

In [ ]:
import torch
from transformers import AutoModel
from bacformer.pp import protein_seqs_to_bacformer_inputs

device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = AutoModel.from_pretrained(
    "macwiatrak/bacformer-masked-complete-genomes",
    trust_remote_code=True
).to(device).eval().to(torch.bfloat16 if device.startswith("cuda") else torch.float32)

inputs = protein_seqs_to_bacformer_inputs(
    genome_info["protein_sequence"],
    device=device,
    batch_size=128,
    max_n_proteins=6000,
)

with torch.no_grad():
    outputs = model(**inputs, return_dict=True)

print("last hidden state shape:", outputs["last_hidden_state"].shape)

In [ ]:
import umap.umap_ as umap
import plotly.express as px

# garder uniquement tokens protéines
stm = inputs["special_tokens_mask"][0].detach().cpu().numpy()
vals, cnts = np.unique(stm, return_counts=True)
prot_id = vals[np.argmax(cnts)]
prot_mask = (stm == prot_id)

H_all = outputs["last_hidden_state"][0].detach().cpu().float().numpy()
H = H_all[prot_mask]

print("proteins only:", H.shape[0], "| genes_df:", len(genes_df))
assert H.shape[0] == len(genes_df)

# UMAP
coords = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(H)
df = genes_df.copy()
df["umap1"] = coords[:, 0]
df["umap2"] = coords[:, 1]

df["partition_norm"] = (
    df["partition"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("", "unannotated")
)

df["color"] = df["partition_norm"]

color_map = {
    "persistent": "orange",
    "shell": "green",
    "cloud": "blue",
    "unannotated": "lightgray",
    "other": "gray",
}

fig = px.scatter(
    df,
    x="umap1",
    y="umap2",
    color="color",
    color_discrete_map=color_map,
    category_orders={"color": ["persistent", "shell", "cloud", "unannotated", "other"]},
    hover_data=["ID","Parent","family","partition","rgp","module","spot","product","seqid","start","end"],
    render_mode="webgl",
    title="UMAP Bacformer — partition (PPanGGOLiN)",
)
fig.update_traces(marker=dict(size=5, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()


In [ ]:
counts = df["partition_norm"].value_counts()

n_persistent = int(counts.get("persistent", 0))
n_shell = int(counts.get("shell", 0))
n_cloud = int(counts.get("cloud", 0))
n_unannotated = int(counts.get("unannotated", 0))

print("Counts:")
print("  persistent :", n_persistent)
print("  shell      :", n_shell)
print("  cloud      :", n_cloud)
print("  unannotated:", n_unannotated)
print("  total      :", len(df))

In [ ]:
out_html = "umap_souche_partition.html"
fig.write_html(out_html, include_plotlyjs="cdn", full_html=True)

from google.colab import files
files.download(out_html)

In [ ]:
import umap.umap_ as umap
import plotly.express as px

# garder uniquement tokens protéines
stm = inputs["special_tokens_mask"][0].detach().cpu().numpy()
vals, cnts = np.unique(stm, return_counts=True)
prot_id = vals[np.argmax(cnts)]
prot_mask = (stm == prot_id)

H_all = outputs["last_hidden_state"][0].detach().cpu().float().numpy()
H = H_all[prot_mask]

print("proteins only:", H.shape[0], "| genes_df:", len(genes_df))
assert H.shape[0] == len(genes_df)

# UMAP
coords = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(H)
df = genes_df.copy()
df["umap1"] = coords[:, 0]
df["umap2"] = coords[:, 1]

COLOR_BY = "family"  # change: "partition" | "rgp" | "module" | "spot"
TOP_K = 50              # None pour tout afficher

series = df[COLOR_BY].fillna("").replace("", "Unannotated")
df["color"] = series if TOP_K is None else series.where(series.isin(series.value_counts().head(TOP_K).index), other="Other")

fig = px.scatter(
    df,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=["ID","Parent","family","partition","rgp","module","spot","product","seqid","start","end"],
    render_mode="webgl",
    title=f"UMAP Bacformer — coloré par {COLOR_BY} (PPanGGOLiN)",
)
fig.update_traces(marker=dict(size=5, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
out_html = f"umap_souche_{COLOR_BY}.html"
fig.write_html(out_html, include_plotlyjs="cdn")

from google.colab import files
files.download(out_html)

In [ ]:
import umap.umap_ as umap
import plotly.express as px

# garder uniquement tokens protéines
stm = inputs["special_tokens_mask"][0].detach().cpu().numpy()
vals, cnts = np.unique(stm, return_counts=True)
prot_id = vals[np.argmax(cnts)]
prot_mask = (stm == prot_id)

H_all = outputs["last_hidden_state"][0].detach().cpu().float().numpy()
H = H_all[prot_mask]

print("proteins only:", H.shape[0], "| genes_df:", len(genes_df))
assert H.shape[0] == len(genes_df)

# UMAP
coords = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(H)
df = genes_df.copy()
df["umap1"] = coords[:, 0]
df["umap2"] = coords[:, 1]

COLOR_BY = "rgp"  # change: "partition" | "family" | "module" | "spot"
TOP_K = 50              # None pour tout afficher

series = df[COLOR_BY].fillna("").replace("", "Unannotated")
df["color"] = series if TOP_K is None else series.where(series.isin(series.value_counts().head(TOP_K).index), other="Other")

fig = px.scatter(
    df,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=["ID","Parent","family","partition","rgp","module","spot","product","seqid","start","end"],
    render_mode="webgl",
    title=f"UMAP Bacformer — coloré par {COLOR_BY} (PPanGGOLiN)",
)
fig.update_traces(marker=dict(size=5, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
out_html = f"umap_souche_{COLOR_BY}.html"
fig.write_html(out_html, include_plotlyjs="cdn")

from google.colab import files
files.download(out_html)

In [ ]:
import umap.umap_ as umap
import plotly.express as px

# garder uniquement tokens protéines
stm = inputs["special_tokens_mask"][0].detach().cpu().numpy()
vals, cnts = np.unique(stm, return_counts=True)
prot_id = vals[np.argmax(cnts)]
prot_mask = (stm == prot_id)

H_all = outputs["last_hidden_state"][0].detach().cpu().float().numpy()
H = H_all[prot_mask]

print("proteins only:", H.shape[0], "| genes_df:", len(genes_df))
assert H.shape[0] == len(genes_df)

# UMAP
coords = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(H)
df = genes_df.copy()
df["umap1"] = coords[:, 0]
df["umap2"] = coords[:, 1]

COLOR_BY = "module"  # change: "partition" | "family" | "rgp" | "spot"
TOP_K = 50              # None pour tout afficher

series = df[COLOR_BY].fillna("").replace("", "Unannotated")
df["color"] = series if TOP_K is None else series.where(series.isin(series.value_counts().head(TOP_K).index), other="Other")

fig = px.scatter(
    df,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=["ID","Parent","family","partition","rgp","module","spot","product","seqid","start","end"],
    render_mode="webgl",
    title=f"UMAP Bacformer — coloré par {COLOR_BY} (PPanGGOLiN)",
)
fig.update_traces(marker=dict(size=5, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
out_html = f"umap_souche_{COLOR_BY}.html"
fig.write_html(out_html, include_plotlyjs="cdn")

from google.colab import files
files.download(out_html)

In [ ]:
import umap.umap_ as umap
import plotly.express as px

# garder uniquement tokens protéines
stm = inputs["special_tokens_mask"][0].detach().cpu().numpy()
vals, cnts = np.unique(stm, return_counts=True)
prot_id = vals[np.argmax(cnts)]
prot_mask = (stm == prot_id)

H_all = outputs["last_hidden_state"][0].detach().cpu().float().numpy()
H = H_all[prot_mask]

print("proteins only:", H.shape[0], "| genes_df:", len(genes_df))
assert H.shape[0] == len(genes_df)

# UMAP
coords = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(H)
df = genes_df.copy()
df["umap1"] = coords[:, 0]
df["umap2"] = coords[:, 1]

COLOR_BY = "spot"  # change: "partition" | "family" | "rgp" | "module"
TOP_K = 0              # None pour tout afficher

series = df[COLOR_BY].fillna("").replace("", "Unannotated")
df["color"] = series if TOP_K is None else series.where(series.isin(series.value_counts().head(TOP_K).index), other="Other")

fig = px.scatter(
    df,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=["ID","Parent","family","partition","rgp","module","spot","product","seqid","start","end"],
    render_mode="webgl",
    title=f"UMAP Bacformer — coloré par {COLOR_BY} (PPanGGOLiN)",
)
fig.update_traces(marker=dict(size=5, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
out_html = f"umap_souche_{COLOR_BY}.html"
fig.write_html(out_html, include_plotlyjs="cdn")

from google.colab import files
files.download(out_html)

In [ ]:
import numpy as np
import pandas as pd

def count_total_and_unique_nonempty(df: pd.DataFrame, col: str):
    if col not in df.columns:
        return 0, 0

    s = (
        df[col]
        .astype(str)
        .str.strip()
        .replace({"": np.nan, "NA": np.nan, "-": np.nan, "nan": np.nan})
    )
    unique_nonempty = int(s.dropna().nunique())    # nb valeurs distinctes non vides
    return unique_nonempty

rgp_unique = count_total_and_unique_nonempty(genes_df, "rgp")
spot_unique = count_total_and_unique_nonempty(genes_df, "spot")
module_unique = count_total_and_unique_nonempty(genes_df, "module")

print(f"RGP   : {rgp_unique}")
print(f"Spot  : {spot_unique}")
print(f"Module: {module_unique}")

In [ ]:
import io

def read_emapper_annotations(path: str) -> pd.DataFrame:
    header = None
    data_lines = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("#"):
                if line.lower().startswith("#query"):
                    header = line.lstrip("#").rstrip("\n").split("\t")
                continue
            if line.strip():
                data_lines.append(line)
    if header is None:
        raise ValueError("Header '#query ...' introuvable dans emapper.annotations.")

    df = pd.read_csv(
        io.StringIO("".join(data_lines)),
        sep="\t",
        names=header,
        dtype=str,
        keep_default_na=False,
    )
    if "query" not in df.columns:
        for c in df.columns:
            if c.lower().strip("#") == "query":
                df = df.rename(columns={c: "query"})
                break
    if "query" not in df.columns:
        raise ValueError("Colonne 'query' introuvable.")
    df["query"] = df["query"].astype(str)
    return df

def norm_id(x: str) -> str:
    x = str(x).split()[0]
    return x.split("|")[-1]

ann = read_emapper_annotations(ANNOT_PATH)
ann["query_norm"] = ann["query"].map(norm_id)

df = genes_df.copy()

merge_key = "faa_id" if "faa_id" in df.columns else "ID"
df["_k"] = df[merge_key].map(norm_id)

df_merged = df.merge(ann.drop(columns=["query"]), left_on="_k", right_on="query_norm", how="left").drop(columns=["_k", "query_norm"])
print("Merged rows:", len(df_merged))
df_merged.head()

In [ ]:
import re
from google.colab import files

coords = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(H)
df_merged["umap1"] = coords[:, 0]
df_merged["umap2"] = coords[:, 1]

def first_item(x: str, sep: str = ",") -> str:
    x = "" if x is None else str(x)
    x = x.strip()
    if not x or x in ("-", "NA"):
        return "Unannotated"
    return x.split(sep)[0].strip()

def safe_filename(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(s))
    return s.strip("_")[:120] or "plot"

def plot_umap(
    field: str,
    sep: str = ",",
    top_k: int | None = 50,
    title: str | None = None,
    export_html: bool = True,
    prefix: str = "umap_bacformer",
):
    if field not in df_merged.columns:
        raise ValueError(f"Colonne '{field}' absente. Colonnes dispo: {df_merged.columns.tolist()}")

    label = df_merged[field].map(lambda x: first_item(x, sep=sep))

    if top_k is None:
        color = label
        topk_tag = "all"
    else:
        keep = set(label.value_counts().head(top_k).index)
        color = label.where(label.isin(keep), other="Other")
        topk_tag = f"top{int(top_k)}"

    fig = px.scatter(
        df_merged.assign(color=color),
        x="umap1",
        y="umap2",
        color="color",
        hover_data=["ID", "faa_id", "family", "partition", "product", field],
        render_mode="webgl",
        title=title or f"UMAP Bacformer — {field}",
    )
    fig.update_traces(marker=dict(size=5, opacity=0.85))
    fig.update_layout(template="plotly_white")
    fig.show()

    if export_html:
        fname = f"{prefix}_{safe_filename(field)}_{topk_tag}.html"
        fig.write_html(fname, include_plotlyjs="cdn", full_html=True)
        files.download(fname)

    return fig

fig_cog = plot_umap("COG_category", sep=",", top_k=50)

if "PFAMs" in df_merged.columns:
    fig_pfam = plot_umap("PFAMs", sep=",", top_k=50)

if "GOs" in df_merged.columns:
    fig_go = plot_umap("GOs", sep=",", top_k=50)

for k in ["KEGG_ko", "KEGG_Pathway", "KEGG_Module"]:
    if k in df_merged.columns:
        fig_kegg = plot_umap(k, sep=",", top_k=50)
        break

In [ ]:
import re
import numpy as np
import pandas as pd
import umap.umap_ as umap
import plotly.express as px

import ipywidgets as widgets
from IPython.display import display
from google.colab import files
from google.colab import output
output.enable_custom_widget_manager()

import plotly.io as pio
pio.renderers.default = "colab"

In [ ]:
# UMAP interactive chooser + Export HTML

# CONFIG
TOP_K_DEFAULT = 50     # 0 show all
SEP_DEFAULT = ","      #(PFAMs/GOs/KEGG often ',' or ';')

def first_item(x: str, sep: str = ",") -> str:
    x = "" if x is None else str(x)
    x = x.strip()
    if not x or x in ("-", "NA"):
        return "Unannotated"
    return x.split(sep)[0].strip()

def build_color_series(df: pd.DataFrame, field: str, sep: str, top_k: int) -> pd.Series:
    s = df[field] if field in df.columns else pd.Series(["Unannotated"] * len(df))
    label = s.map(lambda v: first_item(v, sep=sep))

    if top_k == 0:
        return label
    keep = set(label.value_counts().head(top_k).index)
    return label.where(label.isin(keep), other="Other")

def make_umap_figure(df: pd.DataFrame, field: str, sep: str, top_k: int):
    color = build_color_series(df, field, sep=sep, top_k=top_k)

    hover_cols = [c for c in ["ID", "faa_id", "family", "partition", "rgp", "module", "spot", "product", field] if c in df.columns]

    fig = px.scatter(
        df.assign(color=color),
        x="umap1",
        y="umap2",
        color="color",
        hover_data=hover_cols,
        render_mode="webgl",
        title=f"UMAP Bacformer — color_by={field}",
    )
    fig.update_traces(marker=dict(size=5, opacity=0.85))
    fig.update_layout(template="plotly_white")
    return fig

def safe_filename(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    return s.strip("_")[:120] or "plot"

reserved = {"umap1", "umap2", "protein_sequence"}
fields = [c for c in df_merged.columns if c not in reserved]

preferred = ["COG_category", "PFAMs", "GOs", "KEGG_ko", "KEGG_Pathway", "KEGG_Module"]
fields_sorted = [c for c in preferred if c in fields] + [c for c in fields if c not in preferred]
if not fields_sorted:
    raise ValueError("Aucun champ disponible pour color_by.")

# Widgets
dd = widgets.Dropdown(options=fields_sorted, value=fields_sorted[0], description="color_by:", layout=widgets.Layout(width="520px"))
sep_w = widgets.Text(value=SEP_DEFAULT, description="sep:", layout=widgets.Layout(width="220px"))
topk_w = widgets.IntText(value=TOP_K_DEFAULT, description="top_k (0=all):", layout=widgets.Layout(width="260px"))

btn_show = widgets.Button(description="Afficher", button_style="primary")
btn_export = widgets.Button(description="Exporter HTML", button_style="success")

out = widgets.Output()
state = {"fig": None}

def refresh(_=None):
    with out:
        out.clear_output(wait=True)
        fig = make_umap_figure(df_merged, dd.value, sep=sep_w.value, top_k=int(topk_w.value))
        state["fig"] = fig
        fig.show()

def export_html(_=None):
    fig = state.get("fig")
    if fig is None:
        refresh()
        fig = state.get("fig")

    fname = f"umap_{safe_filename(dd.value)}.html"
    fig.write_html(fname, include_plotlyjs="cdn")
    files.download(fname)

btn_show.on_click(refresh)
btn_export.on_click(export_html)

display(widgets.HBox([dd, topk_w, sep_w, btn_show, btn_export]))
display(out)

refresh()
